# 01 logistic regression

**Owner:** Denis. Original notebook cells: 29, 30, 31, 32, 33, 34, 35, 36.

Run in the shared Databricks session using `run_pipeline.ipynb` or the documented stage runner. This is not a standalone clean-kernel notebook. Saved outputs have been removed for public-repository safety. See `docs/execution_order.md`.

In [ ]:
# Denis: load the shared appointment dataset

import pandas as pd

NO_SHOW_CSV = "/Volumes/workspace/default/eye_style_team2/raw/KaggleV2-May-2016.csv"

appointments = pd.read_csv(NO_SHOW_CSV)

print("Dataset shape:", appointments.shape)
display(appointments.head())

print("\nData types:")
print(appointments.dtypes)

print("\nNo-show class balance:")
print(appointments["No-show"].value_counts(dropna=False))

In [ ]:
# Define the binary prediction target
# 0 = Patient attended the appointment
# 1 = Patient missed the appointment

appointments["target_no_show"] = appointments["No-show"].map({
    "No": 0,
    "Yes": 1
})

# Verify all values were encoded successfully
assert appointments["target_no_show"].notna().all(), \
    "Unexpected target values found"

print("Target encoding:")
print(appointments[["No-show", "target_no_show"]].drop_duplicates())

print("\nClass distribution:")
print(appointments["target_no_show"].value_counts())

print("\nClass percentages:")
print(
    appointments["target_no_show"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
# Initial candidate input features
# Final selection will be reviewed after Daniel's EDA.

candidate_features = [
    "Gender",
    "ScheduledDay",
    "AppointmentDay",
    "Age",
    "Neighbourhood",
    "Scholarship",
    "Hipertension",
    "Diabetes",
    "Alcoholism",
    "Handcap",
    "SMS_received"
]

target_column = "target_no_show"

print("Prediction target:", target_column)
print("\nCandidate features:")
for feature in candidate_features:
    print("-", feature)

# Confirm each proposed column exists
assert all(
    feature in appointments.columns
    for feature in candidate_features
), "One or more candidate features are missing"

print("\nAll candidate features verified.")

### Baseline Model

In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit

# Load Daniel's cleaned dataset
clean_path = "/Volumes/workspace/default/eye_style_team2/clean/cleaned_appointments_features.csv"

model_df = pd.read_csv(clean_path)
print("Cleaned dataset shape:", model_df.shape)

# Only use information available when an appointment is booked.
# SMS_received is excluded because reminder timing is unknown.
# PatientId is used for grouping, not as a model input.

numeric_features = [
    "Age",
    "LeadTime_Days",
    "Scholarship",
    "Hipertension",
    "Diabetes",
    "Alcoholism",
    "Handcap"
]

categorical_features = [
    "Gender",
    "Neighbourhood",
    "Appointment_Weekday"
]

feature_columns = numeric_features + categorical_features
target_column = "target_no_show"

# Verify required columns exist
required_columns = (
    feature_columns +
    [target_column, "PatientId", "AppointmentID"]
)

missing = [
    col for col in required_columns
    if col not in model_df.columns
]

assert not missing, f"Missing columns: {missing}"
assert not model_df[required_columns].isna().any().any(), \
    "Unexpected missing values in modeling data"

# Define features, target and patient groups
X = model_df[feature_columns].copy()
y = model_df[target_column].astype(int)
groups = model_df["PatientId"]

# First split: approximately 70% training, 30% remaining
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, remaining_idx = next(
    splitter.split(X, y, groups=groups)
)

# Second split: divide remaining data into validation and test
remaining_groups = groups.iloc[remaining_idx]

splitter2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_rel_idx, test_rel_idx = next(
    splitter2.split(
        X.iloc[remaining_idx],
        y.iloc[remaining_idx],
        groups=remaining_groups
    )
)

val_idx = remaining_idx[val_rel_idx]
test_idx = remaining_idx[test_rel_idx]

# Create the three datasets
X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

# Confirm there is no patient overlap
train_patients = set(groups.iloc[train_idx])
val_patients = set(groups.iloc[val_idx])
test_patients = set(groups.iloc[test_idx])

assert train_patients.isdisjoint(val_patients)
assert train_patients.isdisjoint(test_patients)
assert val_patients.isdisjoint(test_patients)

# Inspect target balance in each split
for name, labels in [
    ("Training", y_train),
    ("Validation", y_val),
    ("Test", y_test)
]:
    print(
        f"{name}: {len(labels):,} records | "
        f"No-show rate: {labels.mean() * 100:.2f}%"
    )

print("\nFeatures used:", feature_columns)
print("Patient overlap: None")

In [ ]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
import pandas as pd
import time

# 1. Preprocessing
# Numerical features are scaled.
# Categorical features are one-hot encoded.
# Preprocessing is fitted on training data only.

numeric_transformer = Pipeline([
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# 2. Logistic Regression baseline
baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

# 3. Train on training data only
print("Training Logistic Regression baseline...")

start_time = time.time()

baseline_model.fit(X_train, y_train)

training_time = time.time() - start_time

print(f"Training completed in {training_time:.2f} seconds")

# 4. Predict on validation data
val_predictions = baseline_model.predict(X_val)

val_probabilities = baseline_model.predict_proba(X_val)[:, 1]

# 5. Calculate validation metrics
baseline_results = {
    "Accuracy": accuracy_score(y_val, val_predictions),
    "Precision": precision_score(y_val, val_predictions, zero_division=0),
    "Recall": recall_score(y_val, val_predictions, zero_division=0),
    "F1": f1_score(y_val, val_predictions, zero_division=0),
    "ROC-AUC": roc_auc_score(y_val, val_probabilities),
    "PR-AUC": average_precision_score(y_val, val_probabilities)
}

print("\nBASELINE VALIDATION RESULTS: ")

for metric, value in baseline_results.items():
    print(f"{metric}: {value:.4f}")

print("\nCLASSIFICATION REPORT: ")
print(classification_report(
    y_val,
    val_predictions,
    target_names=["Attended", "No-show"],
    zero_division=0
))

print("\nCONFUSION MATRIX: ")
print(confusion_matrix(y_val, val_predictions))

# 6. Save results in a DataFrame
baseline_results_df = pd.DataFrame(
    [baseline_results],
    index=["Logistic Regression"]
)

display(baseline_results_df.round(4))

In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Compare several classification thresholds
thresholds = np.arange(0.20, 0.81, 0.05)
results = []

for threshold in thresholds:
    predictions = (val_probabilities >= threshold).astype(int)

    results.append({
        "Threshold": round(float(threshold), 2),
        "Accuracy": accuracy_score(y_val, predictions),
        "Precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_val, predictions, zero_division=0
        )
    })

threshold_df = pd.DataFrame(results)

print("THRESHOLD COMPARISON: ")
display(threshold_df.round(4))

# Select the threshold with the highest validation F1
best_row = threshold_df.loc[threshold_df["F1"].idxmax()]
best_threshold = float(best_row["Threshold"])

print(f"\nBest validation F1 threshold: {best_threshold:.2f}")
print(f"Validation F1: {best_row['F1']:.4f}")

# Compare against a simple majority-class baseline
majority_predictions = np.zeros(len(y_val), dtype=int)

comparison = pd.DataFrame([
    {
        "Model": "Always predict attendance",
        "Accuracy": accuracy_score(y_val, majority_predictions),
        "Precision": precision_score(
            y_val, majority_predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val, majority_predictions, zero_division=0
        ),
        "F1": f1_score(
            y_val, majority_predictions, zero_division=0
        )
    },
    {
        "Model": "Logistic Regression (0.50)",
        "Accuracy": accuracy_score(y_val, val_predictions),
        "Precision": precision_score(
            y_val, val_predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val, val_predictions, zero_division=0
        ),
        "F1": f1_score(
            y_val, val_predictions, zero_division=0
        )
    },
    {
        "Model": f"Logistic Regression ({best_threshold:.2f})",
        "Accuracy": best_row["Accuracy"],
        "Precision": best_row["Precision"],
        "Recall": best_row["Recall"],
        "F1": best_row["F1"]
    }
])

print("\nBASELINE COMPARISON: ")
display(comparison.round(4))

print("\nConfusion matrix at selected threshold:")
selected_predictions = (
    val_probabilities >= best_threshold
).astype(int)

print(confusion_matrix(y_val, selected_predictions))

In [ ]:

import os
import json
import joblib
import pandas as pd
import sklearn

# Shared model artifact directory
artifact_dir = (
    "/Volumes/workspace/default/"
    "eye_style_team2/models/baseline"
)

os.makedirs(artifact_dir, exist_ok=True)

# Save complete Logistic Regression pipeline,
# including scaling and one-hot encoding
model_path = f"{artifact_dir}/logistic_regression.joblib"

joblib.dump(baseline_model, model_path)

# Preserve split assignments for neural-network comparison
split_assignments = pd.DataFrame({
    "AppointmentID": model_df["AppointmentID"],
    "PatientId": model_df["PatientId"],
    "Split": "unassigned"
})

split_assignments.loc[train_idx, "Split"] = "train"
split_assignments.loc[val_idx, "Split"] = "validation"
split_assignments.loc[test_idx, "Split"] = "test"

assert not (split_assignments["Split"] == "unassigned").any()
assert split_assignments["AppointmentID"].is_unique

split_path = f"{artifact_dir}/dataset_splits.csv"
split_assignments.to_csv(split_path, index=False)

# Save reproducibility information
metadata = {
    "model": "LogisticRegression",
    "random_state": 42,
    "split_strategy": "GroupShuffleSplit by PatientId",
    "features": feature_columns,
    "excluded_feature": "SMS_received",
    "classification_threshold": best_threshold,
    "validation_metrics": {
        key: float(value)
        for key, value in baseline_results.items()
    },
    "sklearn_version": sklearn.__version__
}

metadata_path = f"{artifact_dir}/baseline_metadata.json"

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("Baseline model saved:", model_path)
print("Dataset splits saved:", split_path)
print("Metadata saved:", metadata_path)

print("\nSplit counts:")
print(split_assignments["Split"].value_counts())
